# Migración de aves rapaces sobre Kèköldi, Talamanca
**Proyecto Aplicado 2 · MA-0321 Cálculo Diferencial e Integral · UCR Sede del Sur**
Allison Georgina Monge Navarro · C-06

**Pregunta de decisión:** ¿qué fechas y qué horario debe tener la campaña de voluntariado para registrar la mayor cantidad de aves migratorias?

Este cuaderno sigue la ruta del curso: *datos observados → tasa → integral → verificación → escenarios → decisión*. Toda la matemática está en la carpeta `src/`; aquí se usa y se explica.

> Los datos son simulados con fines didácticos; no son registros oficiales del programa de conteo.

In [ ]:
import pandas as pd

from src import escenarios as es
from src import graficos as gr
from src.datos import cargar_base
from src.limpieza import limpiar_conteos, resumen_limpieza
from src.modelo import (
    CurvaDePaso,
    error_relativo,
    mejor_ventana,
    perfil_horario,
    total_diario,
    total_diario_trapecio,
    totales_diarios,
    verificar_total,
)

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

## 1. Lectura de la base
El Excel se lee directamente (sin copiar valores a mano) y se revisa que tenga las hojas, columnas y unidades esperadas.

In [ ]:
base = cargar_base()
p = base.parametros
conteos = base.conteos
print(p)
conteos.head()

## 2. Exploración: periodo, unidades y patrones
- **Periodo:** 50 días, del 20 de septiembre al 8 de noviembre de 2026.
- **Unidades:** cada registro es el número de aves de una especie contadas en un bloque de una hora (aves); dividido entre 1 h es una **tasa** (aves/h).
- **Horario observado:** bloques de 7:00 a 15:00, es decir, de 7:00 a 16:00. El vuelo real va de h_i a h_f (hoja *Parametros*).

In [ ]:
print(
    "Fechas:",
    conteos["fecha"].min().date(),
    "a",
    conteos["fecha"].max().date(),
    "|",
    conteos["fecha"].nunique(),
    "días",
)
print("Bloques horarios:", sorted(conteos["hora_inicio"].unique()))
print("Registros por especie:")
print(conteos["especie"].value_counts())
conteos.groupby("especie")["aves_contadas"].describe()

### Patrón por clima
La lluvia reduce el vuelo y además es cuando más faltan conteos.

In [ ]:
clima = conteos.groupby("condicion").agg(
    horas=("aves_contadas", "size"),
    horas_sin_conteo=("aves_contadas", lambda s: s.isna().sum()),
    aves_promedio_por_hora=("aves_contadas", "median"),
)
clima

## 3. Faltantes y atípicos
**Criterio (documentado en `docs/bitacora_supuestos.md`):**
1. *Cero de más:* un conteo de al menos 20 aves que es más de 5 veces el promedio de sus horas vecinas del mismo día.
2. *Horas sin conteo:* valores vacíos (NaN), casi todos en horas con lluvia.

Los dos casos se rellenan **interpolando en línea recta dentro del mismo día**. La columna original `aves_contadas` no se modifica; el valor corregido queda en `aves_limpias`.

In [ ]:
limpio = limpiar_conteos(conteos)
display(resumen_limpieza(limpio))
limpio[limpio["correccion"] == "atípico"][
    ["fecha", "hora_inicio", "especie", "condicion", "aves_contadas", "aves_limpias"]
]

In [ ]:
ej = limpio[limpio["correccion"] == "atípico"].sort_values("aves_contadas").iloc[-1]
dia = limpio[(limpio["dia"] == ej["dia"]) & (limpio["especie"] == ej["especie"])]
gr.antes_y_despues(
    dia["aves_contadas"].to_numpy(),
    dia["aves_limpias"].to_numpy(),
    f"Conteo con un cero de más: {ej['especie']}, {ej['fecha']:%d/%m}",
);

## 4. Total diario como integral de la tasa
$$N_d=\int_{h_i}^{h_f} r_d(x)\,dx$$
- Entre 7:00 y 16:00 la tasa se toma constante en cada hora, así que el área de cada bloque es el conteo (aves/h × 1 h = aves).
- Antes de las 7:00 y después de las 16:00 no hay conteo: la tasa se modela como una **recta que llega a cero** en $h_i$ y en $h_f$. Cada cola es un triángulo: $\tfrac12\,(\text{base})(\text{altura})$.

In [ ]:
conteo_dia = dia["aves_limpias"].to_numpy()
bloques = total_diario(conteo_dia, p.hi, p.hf)
trap = total_diario_trapecio(conteo_dia, p.hi, p.hf)
print(f"Bloques + triángulos: {bloques:,.0f} aves")
print(f"Trapecio (puntos a la mitad de cada hora): {trap:,.0f} aves")
print(f"Error relativo entre los dos métodos: {error_relativo(trap, bloques):.2%}")
gr.integral_de_un_dia(conteo_dia, p.hi, p.hf, "Total de un día = área bajo la tasa de paso");

In [ ]:
totales = totales_diarios(limpio, p.hi, p.hf)
contado = limpio["aves_limpias"].sum()
print(f"Aves contadas (7:00-16:00): {contado:,.0f}")
estimado = totales["Total"].sum()
print(f"Aves estimadas con las colas: {estimado:,.0f}  (+{estimado / contado - 1:.1%})")
totales.head()

## 5. La temporada: total, curva acumulada y fechas clave
$$N=\int_{t_0}^{t_1} n(t)\,dt,\qquad F(t)=\frac1N\int_{t_0}^{t} n(u)\,du,\qquad \bar t=\frac1N\int_{t_0}^{t_1} t\,n(t)\,dt$$
Las fechas del 5, 50 y 95 % salen de resolver $F(t)=p$ por interpolación lineal.

In [ ]:
curvas = {e: CurvaDePaso(e, totales["dia"], totales[e]) for e in [*es.ESPECIES, "Total"]}
resumen = pd.DataFrame([c.resumen() for c in curvas.values()]).set_index("especie")
fecha0 = totales["fecha"].min()
for col in ["fecha_5", "fecha_50", "fecha_95", "fecha_media"]:
    resumen[col + "_cal"] = [es.dia_a_fecha(v, fecha0) for v in resumen[col]]
resumen

In [ ]:
gr.curva_acumulada(totales, [*es.ESPECIES, "Total"]);

**Lectura:** el gavilán aludo pasa primero, el gavilán de Swainson a mitad de temporada y el zopilote cabecirrojo al final, con el paso más disperso. En las especies con cola larga hacia un lado, la fecha media y la del 50 % no coinciden: la media se "estira" hacia la cola, pero la mediana no.

### Verificación: integral vs. acumulado observado

In [ ]:
pd.DataFrame(
    [{"serie": e, **verificar_total(totales[e], totales["dia"])} for e in [*es.ESPECIES, "Total"]]
)

El trapecio coincide con la suma de los totales diarios (el acumulado observado) con un error menor que 0,1 %. Simpson se aleja hasta un 3 %, porque supone que la curva es suave (parábolas) y los datos diarios tienen saltos por la lluvia; por eso el modelo usa el trapecio.

## 6. Ventana óptima de la campaña
$$G(a)=\int_a^{a+L} n(t)\,dt,\qquad G'(a)=n(a+L)-n(a)=0$$
Por el **teorema fundamental del cálculo**, la derivada de $G$ es la tasa en el extremo final menos la tasa en el extremo inicial. En el máximo, el primer y el último día de la campaña tienen casi la misma cantidad de aves.

In [ ]:
L = p.dias_campana
r = mejor_ventana(totales["Total"].to_numpy(), L, totales["dia"].to_numpy(float))
desde, hasta = es.dia_a_fecha(r.inicio, fecha0), es.dia_a_fecha(r.fin - 1, fecha0)
print(f"Mejor inicio: día {r.inicio:.0f}, campaña del {desde} al {hasta}")
print(f"Fracción del paso en esas fechas: {r.fraccion:.1%}")
print(es.comprobar_extremos(totales, r.inicio, L))
gr.funcion_G(totales, L, p.inicio_tradicional);

In [ ]:
gr.condicion_optimo(totales, L);

In [ ]:
gr.conteos_diarios(totales, es.ESPECIES, r.inicio, L, p.inicio_tradicional);

## 7. Mejor turno
Con el perfil horario promedio de los días de la campaña se busca el bloque de horas seguidas con más aves (otra integral móvil, ahora en horas).

In [ ]:
perfil = perfil_horario(limpio, dias=(r.inicio, r.fin))
hora_ini, frac_turno = es.fraccion_turno(limpio, r.inicio, L, p.horas_turno, p.hi, p.hf)
turno = f"{es.hora_texto(hora_ini)} a {es.hora_texto(hora_ini + p.horas_turno)}"
print(f"Mejor turno: {turno}  ->  {frac_turno:.1%} de las aves del día")
gr.perfil_y_turno(perfil, hora_ini, p.horas_turno, p.hi, p.hf);

## 8. Escenarios y sensibilidad
1. **Tradicional:** empieza el día indicado en el Excel.
2. **Centrada en el 50 %:** regla simple, se centra en la fecha del 50 % del paso.
3. **Óptima:** el máximo de G(a).
4. **Equilibrada:** maximiza el registro de la especie menos cubierta.

In [ ]:
esc = es.comparar_escenarios(totales, limpio, p)
esc[
    [
        "nombre",
        "fecha_inicio",
        "fecha_fin",
        "hora_inicio",
        "hora_fin",
        "frac_fechas",
        "frac_turno",
        "frac_registrada",
        "aves_registradas",
        *[f"frac_{e}" for e in es.ESPECIES],
    ]
]

In [ ]:
gr.barras_escenarios(esc);

In [ ]:
print("Duración de la campaña:")
display(es.sensibilidad_duracion(totales, p, [7, 14, 21, 28, 35]))
print("Horas del turno:")
display(es.sensibilidad_turno(limpio, p, [2, 3, 4, 5, 6, 7, 8]))
print("Duración del vuelo (h_i, h_f):")
display(es.sensibilidad_vuelo(limpio, p, [-0.5, 0.0, 0.5]))
print("Criterio de limpieza:")
display(es.sensibilidad_limpieza(conteos, p, [3, 5, 10]))

**Lectura de la sensibilidad:** la fecha óptima (8 de octubre) no cambia al mover $h_i$ y $h_f$ ni al cambiar el criterio de limpieza; esos supuestos cambian el total estimado, pero no la decisión. El parámetro que más mueve la decisión es la **duración L** de la campaña: con más días el inicio óptimo se adelanta y la ventaja sobre la campaña tradicional se reduce.

## 9. Recomendación

In [ ]:
print(es.recomendacion(esc))

**Condiciones y limitaciones**
- Supone que el calendario de paso de los próximos años se parece al de esta temporada. Si el paso se adelanta o atrasa, hay que mover la campaña. Se recomienda revisar cada año la fecha del 50 %.
- Los días de lluvia reducen el conteo; la campaña no puede evitar la lluvia, pero sí concentrarse en las semanas de mayor paso.
- La campaña óptima registra menos gavilán aludo que la tradicional, porque esta especie pasa antes. Si el programa quiere priorizar esa especie, conviene la opción *equilibrada*.